# Moirai — Two-Stage Fine-Tuning with Money Loss

> **Stage 1** — NLL pre-training (stabilises weights, 5 epochs).  
> **Stage 2** — Money loss fine-tuning (`MoneyMoiraiFinetune`, 3 epochs).  
> Money loss is the asymmetric cost from forecast error: over-prediction sells back at BR- (selling_price); under-prediction buys at BR+ (buying_price).

In [1]:
import warnings, os, copy, sys
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List
from sklearn.metrics import mean_squared_error
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiForecast, MoiraiFinetune
from uni2ts.model.moirai.module import MoiraiModule
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from training.notebooks.old_notebooks.loss_funcs import money, smape, rmse, mape

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    torch.set_float32_matmul_precision("medium")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")

PyTorch : 2.6.0+cu124
CUDA    : True
GPU     : NVIDIA GeForce RTX 3090
Device  : cuda


In [2]:
def smallest_int_dtype(min_val, max_val, signed=True):
    types = ["int8","int16","int32"] if signed else ["uint8","uint16","uint32"]
    for t in types:
        info = np.iinfo(t)
        if info.min <= min_val <= max_val <= info.max:
            return t
    return "int64" if signed else "uint64"


def optimize_df_for_memory(df):
    meta = {}
    for col in df.columns:
        s = df[col]
        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dt = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dt); meta[col] = {"stored_as": dt, "scale": 1}
        elif pd.api.types.is_float_dtype(s):
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype("float32"); meta[col] = {"stored_as": "float32", "scale": 1}; continue
            decimals = non_null.astype(str).apply(
                lambda x: len(x.split(".")[1].rstrip("0")) if "." in x else 0).max()
            if decimals <= 3:
                scale = 10**decimals
                scaled = np.round(s * scale)
                mn, mx = int(np.nanmin(scaled)), int(np.nanmax(scaled))
                idt = smallest_int_dtype(mn, mx, signed=(mn < 0))
                if np.dtype(idt).itemsize < np.dtype("float32").itemsize:
                    df[col] = scaled.astype(idt); meta[col] = {"stored_as": idt, "scale": scale}; continue
            df[col] = s.astype("float32"); meta[col] = {"stored_as": "float32", "scale": 1}
    return df, meta

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
TRAIN_PATH  = "../../../data/silver_money_calc/train.parquet"
VAL_PATH    = "../../../data/silver_money_calc/val.parquet"
TEST_PATH   = "../../../data/silver_money_calc/test.parquet"
PRICES_PATH = "../../../data/additional_data/ВхідніДані.xlsx"

Y_COL     = "Sum of кВт"
GROUP_COL = "EIC-код_cat"

# ── Moirai ────────────────────────────────────────────────────────────────────
MODEL_ID    = "Salesforce/moirai-1.1-R-small"
CONTEXT_LEN = 168
PRED_LEN    = 48
PATCH_SIZE  = 8
NUM_SAMPLES = 100

# ── Stage 1 — NLL pre-training ────────────────────────────────────────────────
TRAIN_STRIDE = 24
BATCH_SIZE   = 64
NLL_EPOCHS   = 5       # short NLL warm-up for stable weights
LR           = 1e-4
WEIGHT_DECAY = 1e-2
NUM_WORKERS  = 0
CKPT_DIR     = "../../../checkpoints"

# ── Stage 2 — Money loss fine-tuning ─────────────────────────────────────────
MONEY_EPOCHS = 3
MONEY_LR     = 2e-5    # smaller LR to avoid destroying NLL-trained weights
SAVE_PATH    = os.path.join(CKPT_DIR, "moirai_money.pt")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

weather_cols_to_drop = [
    "apparent_temperature", "rain", "snowfall",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "surface_pressure", "wind_direction_10m", "wind_gusts_10m",
    "diffuse_radiation", "direct_normal_irradiance",
]

In [ ]:
GLOBAL_MIN_DT = None

def build_time_index(df):
    global GLOBAL_MIN_DT
    df["datetime"] = pd.to_datetime(df["datetime"])
    if GLOBAL_MIN_DT is None:
        GLOBAL_MIN_DT = df["datetime"].min()
    df["time_idx"] = ((df["datetime"] - GLOBAL_MIN_DT).dt.total_seconds() / 3600).astype(int)
    return df

def load_and_prepare(path, has_y=True):
    df = pd.read_parquet(path).reset_index(drop=True)
    df = df[df["datetime"] >= "2024-06-30"].reset_index(drop=True)
    df.columns = df.columns.str.replace(".", "_", regex=False)
    df, _ = optimize_df_for_memory(df)
    for col in df.select_dtypes(include=["int8","int16","int32","uint8","uint16","uint32"]).columns:
        df[col] = df[col].astype("float32")
    df = build_time_index(df)
    df[Y_COL] = df[Y_COL].astype("float32") if has_y else 0.0
    df = df.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)
    try:
        df.drop(columns=["Ціна розподілу ЕЕ", "Ціна ЕЕ", "Money_spent"], inplace=True)
    except KeyError:
        pass
    df.drop(columns=[c for c in weather_cols_to_drop if c in df.columns], inplace=True)
    return df

In [ ]:
print("Loading train …"); train = load_and_prepare(TRAIN_PATH)
print(f"  train: {train.shape}  {train['datetime'].min()} → {train['datetime'].max()}")
print("Loading val   …"); val  = load_and_prepare(VAL_PATH)
print("Loading test  …"); test = load_and_prepare(TEST_PATH)
print(f"  val: {val.shape}   test: {test.shape}")

In [ ]:
station_stats = (train.groupby(GROUP_COL).agg(rows=(Y_COL,"count"))
                 .reset_index().sort_values("rows", ascending=False))
sampled_stations = station_stats[GROUP_COL].values
train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val  [val  [GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test [test [GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
print(f"Stations: {len(sampled_stations)} | train: {len(train):,}  val: {len(val):,}  test: {len(test):,}")

In [ ]:
train["data_subset"] = "train"
val  ["data_subset"] = "val"
test ["data_subset"] = "test"

all_data = (pd.concat([train, val, test], ignore_index=True)
            .sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True))
all_data["datetime"] = pd.to_datetime(all_data["datetime"], utc=True, errors="coerce").dt.tz_convert(None)
all_data["time_idx"] = ((all_data["datetime"] - all_data["datetime"].min())
                        .dt.total_seconds() // 3600).astype("int64")

training_cutoff = all_data.loc[all_data["data_subset"]=="train", "time_idx"].max()
val_cutoff      = all_data.loc[all_data["data_subset"]=="val",   "time_idx"].max()
test_cutoff     = all_data.loc[all_data["data_subset"]=="test",  "time_idx"].max()

train_mask = all_data["data_subset"] == "train"
all_data.drop(columns=["data_subset"], inplace=True)
train_df = all_data[train_mask].copy()
test_df  = all_data.copy()

print(f"train cutoff: {training_cutoff}  val cutoff: {val_cutoff}  test cutoff: {test_cutoff}")

## Price Lookup
Load day-ahead (РДН), selling (БР-), and buying (БР+) prices — needed both for the money training loss and for evaluation.

In [ ]:
_raw = pd.read_excel(PRICES_PATH, sheet_name="Prices", header=1)
_raw.columns = ["Date/Hour", "selling_price", "buying_price", "day_ahead_price"]
_raw = _raw[_raw["Date/Hour"].notna()].copy()

is_date = _raw["Date/Hour"].astype(str).str.contains("-")
_raw["base_date"] = pd.NaT
_raw.loc[is_date, "base_date"] = pd.to_datetime(_raw.loc[is_date, "Date/Hour"])
_raw["base_date"] = _raw["base_date"].ffill()
_raw["hour"] = pd.to_numeric(_raw["Date/Hour"], errors="coerce")
mask = _raw["hour"].notna()
_raw.loc[mask, "datetime_local"] = (
    _raw.loc[mask, "base_date"] + pd.to_timedelta(_raw.loc[mask, "hour"] - 1, unit="h")
)
prices_raw = _raw[mask][["datetime_local","selling_price","buying_price","day_ahead_price"]].dropna().copy()
prices_raw["datetime_local"] = pd.to_datetime(prices_raw["datetime_local"])

# Fill DST spring-forward gap (2024-03-31 03:00, 2025-03-30 03:00)
_full = pd.date_range(prices_raw["datetime_local"].min(), prices_raw["datetime_local"].max(), freq="h")
prices_raw = (prices_raw.set_index("datetime_local").reindex(_full).ffill()
              .reset_index().rename(columns={"index": "datetime_local"}))
print(f"Price rows: {len(prices_raw):,}  ({prices_raw['datetime_local'].min()} → {prices_raw['datetime_local'].max()})")

In [ ]:
# Convert Ukraine local → UTC-naive (matches all_data['datetime'])
prices_raw["datetime_utc"] = (
    prices_raw["datetime_local"]
    .dt.tz_localize("Europe/Kiev", ambiguous="NaT", nonexistent="NaT")
    .dt.tz_convert("UTC").dt.tz_localize(None)
)
prices_raw = prices_raw.dropna(subset=["datetime_utc"])

global_min_dt = all_data["datetime"].min()
prices_raw["time_idx"] = (
    (prices_raw["datetime_utc"] - global_min_dt).dt.total_seconds() / 3600
).round().astype("int64")

price_lookup = (prices_raw[["time_idx","selling_price","buying_price","day_ahead_price"]]
                .drop_duplicates("time_idx").reset_index(drop=True))
print(f"price_lookup: {len(price_lookup):,} rows, time_idx {price_lookup['time_idx'].min()}–{price_lookup['time_idx'].max()}")
PRICE_COLS = ["selling_price", "buying_price", "day_ahead_price"]

In [ ]:
# Build a dense (max_tidx, 3) numpy array for O(1) price lookup on GPU.
# Columns: 0=selling_price, 1=buying_price, 2=day_ahead_price
_max_tidx = int(price_lookup["time_idx"].max()) + 1
price_arr = np.zeros((_max_tidx, 3), dtype=np.float32)
for _, row in price_lookup.iterrows():
    ti = int(row["time_idx"])
    if 0 <= ti < _max_tidx:
        price_arr[ti] = [row["selling_price"], row["buying_price"], row["day_ahead_price"]]

# Forward-fill any remaining zeros (e.g. time steps before price data begins)
for i in range(1, _max_tidx):
    if price_arr[i].sum() == 0:
        price_arr[i] = price_arr[i - 1]

print(f"price_arr shape: {price_arr.shape}  dtype: {price_arr.dtype}")
print(f"sample (time_idx=0): selling={price_arr[0,0]:.2f}  buying={price_arr[0,1]:.2f}  dap={price_arr[0,2]:.2f}")

In [ ]:
def money_loss_torch(y_pred, y_true, price, selling_price, buying_price):
    """
    Differentiable money loss (asymmetric L1 / pinball structure).
    All tensors must be the same shape. Returns the mean extra cost per sample.
    """
    diff         = y_pred - y_true
    selling_add  = torch.where(diff > 0, selling_price * (y_true - y_pred), torch.zeros_like(diff))
    buying_add   = torch.where(diff < 0, buying_price  * diff,              torch.zeros_like(diff))
    return (selling_add + buying_add).mean()


def compute_money_metrics(eval_df, label):
    """Evaluation helper — attaches prices by time_idx and reports aggregate money cost."""
    df = eval_df.merge(price_lookup[["time_idx"]+PRICE_COLS], on="time_idx", how="left")
    missing = df[PRICE_COLS].isna().any(axis=1).sum()
    if missing > 0:
        print(f"  WARNING: {missing} rows excluded (no price data)")
        df = df.dropna(subset=PRICE_COLS)
    costs = money(df[Y_COL].values, df["pred"].values,
                  df["day_ahead_price"].values, df["selling_price"].values, df["buying_price"].values)
    print(f"{label}")
    print(f"  Total extra cost (UAH)      : {costs.sum():+,.2f}")
    print(f"  Mean extra cost / hour (UAH): {costs.mean():+.4f}")
    return costs

In [ ]:
def build_series_dict(df):
    out = {}
    for eic, grp in df.groupby(GROUP_COL):
        grp = grp.sort_values("time_idx")
        out[eic] = {"values":   grp[Y_COL].values.astype(np.float32),
                    "time_idx": grp["time_idx"].values.astype(np.int64),
                    "datetime": grp["datetime"].values}
    return out

def fill_nan(arr):
    if not np.isnan(arr).any():
        return arr
    return pd.Series(arr).interpolate(method="linear", limit_direction="both").values.astype(np.float32)

train_series = build_series_dict(train_df)
full_series  = build_series_dict(test_df)
lengths = [len(v["values"]) for v in train_series.values()]
print(f"Stations: {len(train_series)}  series len: min={min(lengths)} max={max(lengths)} mean={np.mean(lengths):.0f}")

In [ ]:
class MoiraiWindowDataset(Dataset):
    """
    Sliding-window dataset.  Stores per-window norm params and price time indices
    so the money loss can be computed during training without a separate lookup.
    """
    def __init__(self, series_dict, context_len, pred_len, stride, patch_size):
        self.context_len    = context_len
        self.pred_len       = pred_len
        self.seq_len        = context_len + pred_len
        self.patch_size     = patch_size
        self.n_patches      = self.seq_len // patch_size
        self.n_pred_patches = pred_len // patch_size

        self.windows:        List[np.ndarray] = []   # full (seq_len,) window, unnormalised
        self.pred_time_idxs: List[np.ndarray] = []   # time_idx for prediction steps
        self.pred_targets:   List[np.ndarray] = []   # true kWh for prediction steps

        for eic, data in series_dict.items():
            ts   = fill_nan(data["values"])
            tidx = data["time_idx"]
            if len(ts) < self.seq_len:
                continue
            for s in range(0, len(ts) - self.seq_len + 1, stride):
                self.windows.append(ts[s : s + self.seq_len].copy())
                self.pred_time_idxs.append(tidx[s + context_len : s + self.seq_len].copy())
                self.pred_targets.append(ts[s + context_len : s + self.seq_len].copy())

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        seq = self.windows[idx]               # (seq_len,)  unnormalised
        P, ps = self.n_patches, self.patch_size

        # Per-instance normalisation on context window
        ctx = seq[: self.context_len]
        mu  = float(ctx.mean())
        sig = float(max(ctx.std(), 1e-5))
        seq_n = (seq - mu) / sig

        MAX_PATCH = 128
        target = torch.zeros(P, MAX_PATCH, dtype=torch.float32)
        target[:, :ps] = torch.from_numpy(seq_n.reshape(P, ps)).float()

        observed_mask = torch.zeros(P, MAX_PATCH, dtype=torch.bool)
        observed_mask[:, :ps] = True

        prediction_mask = torch.zeros(P, dtype=torch.bool)
        prediction_mask[-self.n_pred_patches:] = True

        return {
            # ── model inputs ───────────────────────────────────────────────────
            "target":          target,
            "observed_mask":   observed_mask,
            "prediction_mask": prediction_mask,
            "time_id":         torch.arange(P, dtype=torch.long),
            "variate_id":      torch.zeros(P, dtype=torch.long),
            "patch_size":      torch.full((P,), ps, dtype=torch.long),
            "sample_id":       torch.zeros(P, dtype=torch.long),
            # ── money loss extras ──────────────────────────────────────────────
            "pred_time_idx":   torch.from_numpy(self.pred_time_idxs[idx].astype(np.int64)),
            "pred_target_raw": torch.from_numpy(self.pred_targets[idx]).float(),
            "norm_mu":         torch.tensor(mu,  dtype=torch.float32),
            "norm_sig":        torch.tensor(sig, dtype=torch.float32),
        }


def collate_moirai(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


train_ds = MoiraiWindowDataset(train_series, CONTEXT_LEN, PRED_LEN, TRAIN_STRIDE, PATCH_SIZE)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_moirai, num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Training windows : {len(train_ds):,}")
print(f"Batches / epoch  : {len(train_loader):,}")
print(f"Patches / sample : {train_ds.n_patches}  (pred patches: {train_ds.n_pred_patches})")

## Zero-Shot Baseline

In [ ]:
module = MoiraiModule.from_pretrained(MODEL_ID)
zs_model = MoiraiForecast(
    module=module, prediction_length=PRED_LEN, target_dim=1,
    feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
    context_length=CONTEXT_LEN, patch_size=PATCH_SIZE, num_samples=NUM_SAMPLES,
)
zs_predictor = zs_model.create_predictor(batch_size=BATCH_SIZE * 2, device=DEVICE)
print(f"Loaded: {MODEL_ID}  params: {sum(p.numel() for p in zs_model.parameters()):,}")

In [ ]:
def rolling_forecast(predictor, start_cutoff, end_cutoff, split_name="split", infer_batch=256):
    all_windows = []
    for eic in tqdm(full_series, desc=f"Collecting {split_name} windows"):
        data = full_series[eic]
        values = fill_nan(data["values"])
        tidx   = data["time_idx"]
        dts    = data["datetime"]
        tidx_map = {int(t): i for i, t in enumerate(tidx)}
        t = start_cutoff + 1
        while t <= end_cutoff:
            cs = t - CONTEXT_LEN
            if cs not in tidx_map or t not in tidx_map:
                t += PRED_LEN; continue
            s, e = tidx_map[cs], tidx_map[t]
            if e - s != CONTEXT_LEN:
                t += PRED_LEN; continue
            all_windows.append({"eic": eic, "t": t, "ctx": values[s:e],
                                 "ts": values, "tidx_map": tidx_map,
                                 "start": pd.Period(pd.Timestamp(dts[s]), freq="h")})
            t += PRED_LEN

    records = []
    for b0 in tqdm(range(0, len(all_windows), infer_batch), desc=f"Predicting {split_name}"):
        batch = all_windows[b0 : b0 + infer_batch]
        ds = ListDataset([{"target": w["ctx"], "start": w["start"], "item_id": w["eic"]} for w in batch], freq="h")
        for w, fc in zip(batch, predictor.predict(ds)):
            for step, pred_val in enumerate(fc.mean):
                ti = w["t"] + step
                if ti not in w["tidx_map"]: break
                pos = w["tidx_map"][ti]
                records.append({GROUP_COL: w["eic"], "time_idx": int(ti),
                                Y_COL: float(w["ts"][pos]), "pred": float(pred_val)})
    return pd.DataFrame(records)

In [ ]:
zs_val  = rolling_forecast(zs_predictor, training_cutoff, val_cutoff,  "val")
zs_test = rolling_forecast(zs_predictor, val_cutoff,      test_cutoff, "test")
print(f"ZS val rows: {len(zs_val):,}  test rows: {len(zs_test):,}")

In [ ]:
print("── Zero-shot · Val ──")
print(f"SMAPE:{smape(zs_val[Y_COL],zs_val['pred']):.4f}  RMSE:{rmse(zs_val[Y_COL],zs_val['pred']):.4f}  MAPE:{mape(zs_val[Y_COL],zs_val['pred']):.2f}%")
zs_val_costs  = compute_money_metrics(zs_val,  "Money · ZS · Val")
print("── Zero-shot · Test ──")
print(f"SMAPE:{smape(zs_test[Y_COL],zs_test['pred']):.4f}  RMSE:{rmse(zs_test[Y_COL],zs_test['pred']):.4f}  MAPE:{mape(zs_test[Y_COL],zs_test['pred']):.2f}%")
zs_test_costs = compute_money_metrics(zs_test, "Money · ZS · Test")

## Stage 1 — NLL Pre-Training
Short NLL warm-up (`NLL_EPOCHS=5`) to adapt Moirai to the electricity domain before the money loss stage.

In [ ]:
# MoiraiFinetune only uses the standard model-input keys; extra keys in the batch are ignored
# because Lightning passes **batch to training_step, which only reads expected keys.
nll_steps   = len(train_loader) * NLL_EPOCHS
nll_warmup  = min(500, nll_steps // 10)

nll_model = MoiraiFinetune(
    module=copy.deepcopy(module),
    min_patches=4, min_mask_ratio=0.15, max_mask_ratio=0.5, max_dim=1,
    num_training_steps=nll_steps, num_warmup_steps=nll_warmup,
    context_length=CONTEXT_LEN, prediction_length=PRED_LEN, patch_size=PATCH_SIZE,
    lr=LR, weight_decay=WEIGHT_DECAY,
)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"NLL model ready  params: {sum(p.numel() for p in nll_model.parameters()):,}")
print(f"NLL steps: {nll_steps}  warmup: {nll_warmup}")

In [ ]:
nll_ckpt = ModelCheckpoint(
    dirpath=CKPT_DIR, filename="moirai-nll-{epoch:02d}",
    monitor="train/PackedNLLLoss", mode="min", save_top_k=1,
)
nll_trainer = L.Trainer(
    max_epochs=NLL_EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu", devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32",
    callbacks=[nll_ckpt, TQDMProgressBar(refresh_rate=10)],
    log_every_n_steps=10,
)
nll_trainer.fit(nll_model, train_loader)
print(f"NLL best ckpt : {nll_ckpt.best_model_path}")

## Stage 2 — Money Loss Fine-Tuning

`MoneyMoiraiFinetune` subclasses `MoiraiFinetune` and overrides **`training_step`** to:
1. Mask future patches from the model input (so the model must predict, not copy).
2. Call `self.module(...)` to get the predicted distribution.
3. Denormalise the predicted mean using the per-window μ/σ stored in the dataset.
4. Look up prices by `time_idx` from the dense `price_arr` GPU tensor.
5. Compute and return `money_loss_torch`.

In [ ]:
# Keys from the batch that belong to the model (not money-loss extras)
_MODEL_KEYS = ["target", "observed_mask", "prediction_mask",
               "time_id", "variate_id", "patch_size", "sample_id"]


class MoneyMoiraiFinetune(MoiraiFinetune):
    """
    MoiraiFinetune with the NLL training_step replaced by the money loss.
    All Lightning infrastructure (optimizer, scheduler, device placement) is inherited.
    """

    def __init__(self, price_arr: np.ndarray, n_pred_patches: int, patch_size: int, **kwargs):
        super().__init__(**kwargs)
        # Register as buffer → moves to the correct device automatically
        self.register_buffer("_price_tensor", torch.from_numpy(price_arr))
        self._n_pred = n_pred_patches
        self._ps     = patch_size

    def training_step(self, batch, batch_idx):
        # ── Money-loss extras (not passed to the model) ────────────────────────
        pred_time_idx   = batch["pred_time_idx"]    # (B, pred_len)
        pred_target_raw = batch["pred_target_raw"]  # (B, pred_len)
        norm_mu         = batch["norm_mu"]           # (B,)
        norm_sig        = batch["norm_sig"]          # (B,)
        pred_mask       = batch["prediction_mask"]   # (B, P)

        # ── Mask future patches so the model cannot see ground-truth future ────
        target_in = batch["target"].clone()           # (B, P, MAX_PATCH)
        obs_in    = batch["observed_mask"].clone()    # (B, P, MAX_PATCH)
        target_in[pred_mask] = 0.0
        obs_in[pred_mask]    = False

        # ── Forward through MoiraiModule ───────────────────────────────────────
        # MoiraiModule.forward signature (uni2ts v2):
        #   target, observed_mask, time_id, variate_id,
        #   prediction_mask (opt), patch_size (opt), sample_id (opt)
        # Returns a torch.distributions.Distribution whose .mean has shape
        # (B, P, MAX_PATCH) or (B, 1, P, MAX_PATCH) depending on version.
        distr = self.module(
            target          = target_in,
            observed_mask   = obs_in,
            time_id         = batch["time_id"],
            variate_id      = batch["variate_id"],
            prediction_mask = pred_mask,
            patch_size      = batch["patch_size"],
            sample_id       = batch["sample_id"],
        )

        # ── Extract mean for prediction patches ────────────────────────────────
        mean = distr.mean
        if mean.dim() == 4:          # (B, var, P, MAX_PATCH) → squeeze var dim
            mean = mean.squeeze(1)
        # (B, n_pred_patches, patch_size) → (B, pred_len)
        pred_norm = mean[:, -self._n_pred:, :self._ps].reshape(mean.size(0), -1)

        # ── Denormalise ────────────────────────────────────────────────────────
        pred = pred_norm * norm_sig.unsqueeze(1) + norm_mu.unsqueeze(1)

        # ── Price lookup (O(1) from dense GPU tensor) ──────────────────────────
        tidxs  = pred_time_idx.clamp(0, self._price_tensor.size(0) - 1)  # (B, pred_len)
        prices = self._price_tensor[tidxs]          # (B, pred_len, 3)
        # price_arr columns: 0=selling_price, 1=buying_price, 2=day_ahead_price

        # ── Money loss ─────────────────────────────────────────────────────────
        loss = money_loss_torch(
            pred, pred_target_raw,
            prices[..., 2],   # day_ahead_price
            prices[..., 0],   # selling_price
            prices[..., 1],   # buying_price
        )
        self.log("train/money_loss", loss, on_step=True, prog_bar=True)
        return loss

In [ ]:
money_steps  = len(train_loader) * MONEY_EPOCHS
money_warmup = min(200, money_steps // 10)

money_model = MoneyMoiraiFinetune(
    price_arr     = price_arr,
    n_pred_patches= train_ds.n_pred_patches,
    patch_size    = PATCH_SIZE,
    # MoiraiFinetune kwargs — inherit same architecture
    module        = nll_model.module,   # start from NLL-pretrained weights
    min_patches   = 4,
    min_mask_ratio= 0.15,
    max_mask_ratio= 0.5,
    max_dim       = 1,
    num_training_steps = money_steps,
    num_warmup_steps   = money_warmup,
    context_length     = CONTEXT_LEN,
    prediction_length  = PRED_LEN,
    patch_size         = PATCH_SIZE,
    lr           = MONEY_LR,
    weight_decay = WEIGHT_DECAY,
)
print(f"MoneyMoiraiFinetune ready  steps: {money_steps}  warmup: {money_warmup}")

In [ ]:
money_ckpt = ModelCheckpoint(
    dirpath=CKPT_DIR, filename="moirai-money-{epoch:02d}-{train_money_loss:.4f}",
    monitor="train/money_loss", mode="min", save_top_k=1,
)
money_trainer = L.Trainer(
    max_epochs=MONEY_EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu", devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32",
    callbacks=[money_ckpt, TQDMProgressBar(refresh_rate=10)],
    log_every_n_steps=10,
)
money_trainer.fit(money_model, train_loader)
print(f"Money best ckpt : {money_ckpt.best_model_path}")

In [ ]:
torch.save(money_model.module.state_dict(), SAVE_PATH)
print(f"Saved money-trained weights → {SAVE_PATH}")

## Evaluation — Money-Trained Model vs Zero-Shot

In [ ]:
# Load money-trained weights into the inference wrapper
zs_model.module.load_state_dict(money_model.module.state_dict())
ft_predictor = zs_model.create_predictor(batch_size=BATCH_SIZE * 2, device=DEVICE)
print("Money-trained predictor ready")

In [ ]:
val_eval  = rolling_forecast(ft_predictor, training_cutoff, val_cutoff,  "val")
test_eval = rolling_forecast(ft_predictor, val_cutoff,      test_cutoff, "test")
print(f"val: {len(val_eval):,}  test: {len(test_eval):,}")

In [ ]:
print("── Money-trained · Val ──")
print(f"SMAPE:{smape(val_eval[Y_COL],val_eval['pred']):.4f}  RMSE:{rmse(val_eval[Y_COL],val_eval['pred']):.4f}  MAPE:{mape(val_eval[Y_COL],val_eval['pred']):.2f}%")
ft_val_costs  = compute_money_metrics(val_eval,  "Money · Finetuned · Val")

print("── Money-trained · Test ──")
print(f"SMAPE:{smape(test_eval[Y_COL],test_eval['pred']):.4f}  RMSE:{rmse(test_eval[Y_COL],test_eval['pred']):.4f}  MAPE:{mape(test_eval[Y_COL],test_eval['pred']):.2f}%")
ft_test_costs = compute_money_metrics(test_eval, "Money · Finetuned · Test")

print("\n── Δ vs zero-shot (test) ──")
print(f"ΔSMAPE : {smape(test_eval[Y_COL],test_eval['pred']) - smape(zs_test[Y_COL],zs_test['pred']):+.4f}")
print(f"ΔRMSE  : {rmse( test_eval[Y_COL],test_eval['pred']) - rmse( zs_test[Y_COL],zs_test['pred']):+.4f}")
print(f"ΔMoney : {ft_test_costs.sum() - zs_test_costs.sum():+,.2f} UAH total extra cost")

In [ ]:
def per_station_metrics(eval_df):
    df = eval_df.merge(price_lookup[["time_idx"]+PRICE_COLS], on="time_idx", how="left").dropna(subset=PRICE_COLS)
    rows = []
    for grp, gdf in df.groupby(GROUP_COL):
        costs = money(gdf[Y_COL].values, gdf["pred"].values,
                      gdf["day_ahead_price"].values, gdf["selling_price"].values, gdf["buying_price"].values)
        rows.append({GROUP_COL: grp, "n": len(gdf),
                     "SMAPE": smape(gdf[Y_COL], gdf["pred"]),
                     "RMSE":  rmse (gdf[Y_COL], gdf["pred"]),
                     "money_total":    costs.sum(),
                     "money_per_hour": costs.mean()})
    return pd.DataFrame(rows).sort_values("money_total")

test_station_metrics = per_station_metrics(test_eval)
print("Top-10 cheapest:")
print(test_station_metrics.head(10).to_string(index=False))
print("\nTop-10 most expensive:")
print(test_station_metrics.tail(10).to_string(index=False))

In [ ]:
def plot_forecast(eval_df, eic_code, start_dt=None, end_dt=None, title_prefix=""):
    df = eval_df.merge(all_data[[GROUP_COL,"time_idx","datetime"]], on=[GROUP_COL,"time_idx"], how="inner")
    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if start_dt: df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt:   df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      lw=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", lw=1, color="tomato", alpha=0.85)
    ax.set_title(f"{title_prefix}{eic_code}  SMAPE={smape(df[Y_COL],df['pred']):.3f}  RMSE={rmse(df[Y_COL],df['pred']):.2f}")
    ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center"); plt.tight_layout(); plt.show()

plot_forecast(test_eval, test_station_metrics.iloc[0][GROUP_COL],  title_prefix="[BEST money] ")
plot_forecast(test_eval, test_station_metrics.iloc[-1][GROUP_COL], title_prefix="[WORST money] ")